In [ ]:
"""
Author: Emily Vosper + Ruby Lieber
Affiliation: University of Bristol
Date: 26/06/2025

Description:
This script plots heat-related mortality attributable to human-induced climate change
across Brazilian state capitals between 2000 and 2019, using factual and counterfactual
simulations from a multi-model ensemble.
"""

In [ ]:
import os
os.environ["PROJ_DATA"] = "/user/home/al18709/breathe/.pixi/envs/default/share/proj"
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import geopandas as gpd
import seaborn as sns
from dask.distributed import Client, LocalCluster
from dask_jobqueue import SLURMCluster
import pickle
import copy
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def prepare_igr_geodataframe(results, geometry_path):
    """
    Merge IGR-level results with geometries for plotting.
    
    results      : the results dict from compute_summary
    geometry_path: path to shapefile/parquet with IGR geometries and 'rgi' column
    """
    geometry_gdf = gpd.read_file(geometry_path)
    geometry_gdf['rgi'] = geometry_gdf['rgi'].astype(str)

    # Pivot the leaf DataFrame from long (statistic as index) to wide (one column per statistic)
    igr_rows = []
    for igr, df in results['igr'].items():
        row = df['mean'].to_dict()
        row['p2.5']  = df['p2.5'].to_dict()
        row['p97.5'] = df['p97.5'].to_dict()
        row['rgi']   = str(igr)
        igr_rows.append(row)
    
    # Cleaner approach — stack into wide format directly
    igr_rows = []
    for igr, df in results['igr'].items():
        row = {'rgi': str(igr)}
        for statistic in df.index:
            # row[f'{statistic}_mean']  = df.loc[statistic, 'mean']
            # row[f'{statistic}_p2.5']  = df.loc[statistic, 'p2.5']
            # row[f'{statistic}_p97.5'] = df.loc[statistic, 'p97.5']
            row[f'{statistic}_mean']  = float(np.array(df.loc[statistic, 'mean']).item())
            row[f'{statistic}_p2.5']  = float(np.array(df.loc[statistic, 'p2.5']).item())
            row[f'{statistic}_p97.5'] = float(np.array(df.loc[statistic, 'p97.5']).item())
        igr_rows.append(row)

    results_df = pd.DataFrame(igr_rows)
    results_gdf = geometry_gdf.merge(results_df, on='rgi', how='left')

    return results_gdf

In [ ]:
def annotate_capitals(ax, cities, fontsize=18):
    """Scatter and label Brazilian state capitals on a map axis."""
    for city in cities:
        marker = city.get('marker', 'o')
        s = city.get('s', 14)
        ax.scatter(city['x'], city['y'], color='black', s=s, marker=marker,zorder=1000)
        ax.annotate(city['name'], xy=(city['ax'], city['ay']), xycoords='data', fontsize=fontsize)


In [ ]:
def add_region_inset(fig, macro, region_color_map, position):
    """Add a small colour-coded region inset map to the figure."""
    ax_inset = fig.add_axes(position)
    for region in macro.index:
        color = region_color_map.get(region, 'lightgrey')
        macro.loc[[region]].plot(ax=ax_inset, color=color, edgecolor='white', alpha=0.4)
    ax_inset.axis('off')
    for region, row in macro.iterrows():
        centroid = row.geometry.centroid
        ax_inset.text(
            centroid.x, centroid.y, region,
            color=region_color_map.get(region, 'black'),
            fontsize=14, weight='bold', ha='center', va='center'
        )
    return ax_inset

In [ ]:
def annotate_capitals(ax, cities, fontsize=18):
    """Scatter and label Brazilian state capitals on a map axis."""
    for city in cities:
        marker = city.get('marker', 'o')
        s = city.get('s', 14)
        ax.scatter(city['x'], city['y'], color='black', s=s, marker=marker,zorder=1000)
        ax.annotate(city['name'], xy=(city['ax'], city['ay']), xycoords='data', fontsize=fontsize)

In [ ]:
def plot_brazil_ratio(ax, fig, results_gdf, statistic='moderate_high_ratio_mean',
                      title=None, show_cbar=True, cbar_ax=None,fontsize=18,show_cities=True):

    col = f'{statistic}'

    # --- Build custom colormap: purple for <1, orange for >1 ---
    puor = plt.get_cmap('PuOr', 512)
    purple_colors = puor(np.linspace(0.55, 1.0, 256))   # purple end
    orange_colors = puor(np.linspace(0.0, 0.45, 256))   # orange end
    combined      = np.vstack([purple_colors[::-1], orange_colors[::-1]])
    custom_cmap   = mcolors.LinearSegmentedColormap.from_list('PuOr_ratio', combined)
    custom_cmap.set_bad("#2E2E2E")
    custom_cmap.set_under("#2E2E2E")

    # --- Levels: fine 0–1, coarse 1–10 ---
    bounds_low  = np.arange(0, 1.1, 0.1)        # 0.0, 0.1, ..., 1.0
    bounds_high = np.arange(2, 11, 1)            # 2, 3, ..., 10
    bounds      = np.concatenate([bounds_low, bounds_high])  # skip duplicate 1.0→2

    norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=10)

    results_gdf.plot(
        column=col, ax=ax, cmap=custom_cmap, norm=norm,
        legend=False,
        edgecolor="#ffffff", linewidth=0.2, alpha=0.9,
        missing_kwds={"color": "#dfdfdf", "edgecolor": "#dfdfdf", "linewidth": 0.2}
    )

    sm    = plt.cm.ScalarMappable(cmap=custom_cmap, norm=norm)
    sm._A = []

    if show_cbar:
        if cbar_ax is None:
            # make cbar longer
            # cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.015])
            cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.015])

        cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',
                            pad=0, extend='both')

        # Set ticks at all our bounds
        cbar.set_ticks(bounds)

        # Label: 1 decimal for 0–1, integer for 1–10
        tick_labels = [f'{v:.1f}' for v in bounds_low] + [f'{int(v)}' for v in bounds_high]
        cbar.set_ticklabels(tick_labels)

        # White line at vcenter=1
        # Find where 1.0 sits in normalised (0–1) axes coords
        center_pos = norm(1.0)
        # cbar.ax.axvline(x=center_pos, color='white', linewidth=2)

        # Annotate the two sides
        cbar.ax.text(
            0.5, -2.2, 'High > Moderate',
            ha='center', va='top', fontsize=16,
            transform=cbar.ax.transData, color='black'
        )
        cbar.ax.text(
            5, -2.2, 'Moderate > High',
            ha='center', va='top', fontsize=16,
            transform=cbar.ax.transData, color='black'
        )

        cbar.set_label("Ratio (moderate / high exposure)", fontsize=20, labelpad=10)
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(labelsize=14)

    if title is not None:
        ax.set_title(title, fontsize=22, pad=14)

    for spine in ["left", "right", "top", "bottom"]:
        ax.spines[spine].set_visible(False)
    ax.set_yticks([])
    ax.set_xticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    
    if show_cities:
        annotate_capitals(ax, cities,fontsize)

    return ax, fig, sm, bounds

In [ ]:
def load_results(path):
    with open(path, 'rb') as f:
        results = pickle.load(f)
    return results

In [ ]:
def generate_weights(models, simulation_values=None, filter_nans=False):
    merged = copy.deepcopy({k: v for k, v in models.items() if not (k == 'CanESM5-P2' and 'CanESM5' in models)})
    
    if 'CanESM5' in models and 'CanESM5-P2' in models:
        merged['CanESM5']['runs'] = models['CanESM5']['runs'] + models['CanESM5-P2']['runs']
    
    n_models = len(merged)  # 13

    if not filter_nans or simulation_values is None:
        return np.concatenate([
            np.full(len(m['runs']) * 1000, 1 / (n_models * len(m['runs']) * 1000))
            for m in merged.values()
        ])

    # filter_nans=True: weight by non-nan count per model, then zero out nans
    weights_parts = []
    idx = 0
    for m in merged.values():
        model_size  = len(m['runs']) * 1000
        model_vals  = simulation_values[idx : idx + model_size]
        # n_not_nan   = (~np.isnan(model_vals)).sum()
        n_not_nan   = (np.isfinite(model_vals)).sum()
        print('infinite count:', (~np.isfinite(model_vals)).sum())
        print('model:', m['prefix'])
        w_per_sample = 1 / (n_models * n_not_nan) if n_not_nan > 0 else 0
        weights_parts.append(np.full(model_size, w_per_sample))
        idx += model_size

    weights_out = np.concatenate(weights_parts)
    weights_out[~np.isfinite(simulation_values)] = 0
    return weights_out

In [ ]:
def weighted_percentile(data, weights, percentiles):
    i = np.argsort(data)
    d_sorted = data[i]
    w_sorted = weights[i]
    lower = np.percentile(d_sorted,percentiles[0],method='inverted_cdf',weights=w_sorted)
    upper = np.percentile(d_sorted,percentiles[1],method='inverted_cdf',weights=w_sorted)
    return lower,upper

In [ ]:
# ============================================================
# DATA LOADING
# ============================================================

# Load shapefiles
polys       = '/bp1/geog-tropical/data/BREATHE/RG2017_rgi_20180911/RG2017_rgi.shp'
gdf         = gpd.read_file(polys)
polys_states = '/bp1/geog-tropical/data/BREATHE/shapefiles/BR_UF_2022.shp'
gdf_states  = gpd.read_file(polys_states)

# Build macroregion polygons from state boundaries
macroregion_map = {
    'RO': 'North',    'AC': 'North',     'AM': 'North',       'RR': 'North',
    'PA': 'North',    'AP': 'North',     'TO': 'North',
    'MA': 'Northeast','PI': 'Northeast', 'CE': 'Northeast',   'RN': 'Northeast',
    'PB': 'Northeast','PE': 'Northeast', 'AL': 'Northeast',   'SE': 'Northeast', 'BA': 'Northeast',
    'MT': 'Central-West','MS': 'Central-West','GO': 'Central-West','DF': 'Central-West',
    'SP': 'Southeast','RJ': 'Southeast', 'MG': 'Southeast',   'ES': 'Southeast',
    'PR': 'South',    'SC': 'South',     'RS': 'South'
}
gdf_states["macroregion"] = gdf_states["SIGLA_UF"].map(macroregion_map)
macro = gdf_states.dissolve(by="macroregion")

# Load both result sets
results_100m = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M.pkl')
results_low  = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M_lowp95.pkl')
results_high  = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M_highp95.pkl')
results_ratio = load_results('/bp1/geog-tropical/data/BREATHE/results/results_100M_mod_high_ratio_p95.pkl')

# Merge both with geometries
results_gdf_100m = prepare_igr_geodataframe(results_100m, geometry_path=polys)
results_gdf_low  = prepare_igr_geodataframe(results_low,  geometry_path=polys)
results_gdf_high = prepare_igr_geodataframe(results_high, geometry_path=polys)
results_gdf_ratio = prepare_igr_geodataframe(results_ratio, geometry_path=polys)

# Derive region from rgi prefix
uf_code_to_region = {
    11: 'North', 12: 'North', 13: 'North', 14: 'North', 15: 'North', 16: 'North', 17: 'North',
    21: 'Northeast', 22: 'Northeast', 23: 'Northeast', 24: 'Northeast', 25: 'Northeast',
    26: 'Northeast', 27: 'Northeast', 28: 'Northeast', 29: 'Northeast',
    31: 'Southeast', 32: 'Southeast', 33: 'Southeast', 35: 'Southeast',
    41: 'South', 42: 'South', 43: 'South',
    50: 'Central-West', 51: 'Central-West', 52: 'Central-West', 53: 'Central-West',
}
for gdf_ in [results_gdf_100m, results_gdf_low, results_gdf_high, results_gdf_ratio]:
    gdf_['Region'] = gdf_['rgi'].apply(
        lambda x: uf_code_to_region.get(int(str(x)[:2]), None)
    )

# Convenience alias (most plots use 100M)
results_gdf = results_gdf_100m

# Split into present/missing for plotting
df_merged   = results_gdf.dropna(subset=['attributable_mortality_per_100k_mean'])
gdf_missing = results_gdf[results_gdf['attributable_mortality_per_100k_mean'].isna()]

# RGI-to-region lookup for the 27 state capitals
rgi_region_map = {
    '120001': "North",        '270001': "Northeast",   '160001': "North",
    '130001': "North",        '290001': "Northeast",   '230001': "Northeast",
    '530001': "Central-West", '320001': "Southeast",   '520001': "Central-West",
    '210001': "Northeast",    '510001': "Central-West",'500001': "Central-West",
    '310001': "Southeast",    '150001': "North",       '250001': "Northeast",
    '410001': "South",        '260001': "Northeast",   '220001': "Northeast",
    '330001': "Southeast",    '240001': "Northeast",   '430001': "South",
    '110001': "North",        '140001': "North",       '420001': "South",
    '350001': "Southeast",    '280001': "Northeast",   '170001': "North"
}



In [ ]:
# define initial variables

models = {
        'ACCESS-CM2':      {'prefix': 'accesscm2', 'runs': [f'r{r}i1p1f1' for r in [1]]},
        'ACCESS-ESM1-5':   {'prefix': 'access',    'runs': [f'r{r}i1p1f1' for r in range(1, 4)]},
        'BCC-CSM2-MR':     {'prefix': 'bcc',       'runs': [f'r{r}i1p1f1' for r in [1]]},
        'CanESM5':         {'prefix': 'canesm',    'runs': [f'r{r}i1p1f1' for r in range(1, 26) if r not in [12, 16, 17, 18, 19, 20, 21, 24]]},
        'CanESM5-P2':      {'prefix': 'canesmp2',  'runs': [f'r{r}i1p2f1' for r in range(1, 26) if r not in [3, 7, 12, 13, 14, 15, 17, 19, 23]]},
        'CNRM-CM6-1':      {'prefix': 'cnrm',      'runs': [f'r{r}i1p1f2' for r in range(1, 4)]},
        'FGOALS-g3':       {'prefix': 'fgoals',    'runs': [f'r{r}i1p1f1' for r in [1]]},
        'GFDL-CM4':        {'prefix': 'gfdl',      'runs': [f'r{r}i1p1f1' for r in [1]]},
        'GFDL-ESM4':       {'prefix': 'gfdlesm',   'runs': [f'r{r}i1p1f1' for r in [1]]},
        'HadGEM3-GC31-LL': {'prefix': 'hadgem',    'runs': [f'r{r}i1p1f3' for r in range(1, 61) if r not in [6, 7, 8, 9, 10]]},
        'IPSL-CM6A-LR':    {'prefix': 'ipsl',      'runs': [f'r{r}i1p1f1' for r in range(1, 7) if r != 3]},
        'MIROC6':          {'prefix': 'miroc',     'runs': [f'r{r}i1p1f1' for r in range(1, 3)]},
        'MRI-ESM2-0':      {'prefix': 'mri',       'runs': [f'r{r}i1p1f1' for r in range(1, 6) if r != 4]},
        'NorESM2-LM':      {'prefix': 'noresm',    'runs': [f'r{r}i1p1f1' for r in range(1, 4)]},
    }

cities = [
    {'name': 'Rio Branco',     'x': -67.809, 'y':  -9.974, 'ax': -67.4,        'ay': -10.6},
    {'name': 'Maceió',         'x': -35.735, 'y':  -9.666, 'ax': -35.3,        'ay': -10.2},
    {'name': 'Macapá',         'x': -51.050, 'y':   0.034, 'ax': -50.6,        'ay':   0.0},
    {'name': 'Manaus',         'x': -60.021, 'y':  -3.122, 'ax': -59.6,        'ay':  -3.9},
    {'name': 'Salvador',       'x': -38.505, 'y': -12.986, 'ax': -38.2,        'ay': -13.6},
    {'name': 'Fortaleza',      'x': -38.529, 'y':  -3.724, 'ax': -38.2,        'ay':  -3.4},
    {'name': 'Brasília',       'x': -47.895, 'y': -15.807, 'ax': -49.9,        'ay': -15.3, 'marker': 's'},
    {'name': 'Vitória',        'x': -40.308, 'y': -20.315, 'ax': -39.9,        'ay': -21.0},
    {'name': 'Goiânia',        'x': -49.264, 'y': -16.686, 'ax': -48.9,        'ay': -17.4},
    {'name': 'São Luís',       'x': -44.302, 'y':  -2.530, 'ax': -44.0,        'ay':  -2.4},
    {'name': 'Cuiabá',         'x': -56.098, 'y': -15.601, 'ax': -55.7,        'ay': -16.1},
    {'name': 'Campo Grande',   'x': -54.620, 'y': -20.469, 'ax': -59.2,        'ay': -20.7},
    {'name': 'Belo Horizonte', 'x': -43.938, 'y': -19.920, 'ax': -43.5,        'ay': -19.8},
    {'name': 'Belém',          'x': -48.503, 'y':  -1.455, 'ax': -48.0,        'ay':  -2.1},
    {'name': 'João Pessoa',    'x': -34.864, 'y':  -7.121, 'ax': -34.4,        'ay':  -7.35},
    {'name': 'Curitiba',       'x': -49.265, 'y': -25.428, 'ax': -48.8,        'ay': -26.2},
    {'name': 'Recife',         'x': -34.883, 'y':  -8.059, 'ax': -34.5,        'ay':  -8.7},
    {'name': 'Teresina',       'x': -42.803, 'y':  -5.090, 'ax': -45.4,        'ay':  -5.8},
    {'name': 'Rio de Janeiro', 'x': -43.189, 'y': -22.917, 'ax': -42.8,        'ay': -23.7},
    {'name': 'Natal',          'x': -35.209, 'y':  -5.795, 'ax': -34.8,        'ay':  -6.0},
    {'name': 'Porto Alegre',   'x': -51.211, 'y': -30.038, 'ax': -50.8,        'ay': -30.8},
    {'name': 'Porto Velho',    'x': -63.903, 'y':  -8.761, 'ax': -63.5,        'ay':  -9.4},
    {'name': 'Boa Vista',      'x': -60.673, 'y':   2.823, 'ax': -60.2,        'ay':   2.9},
    {'name': 'Florianópolis',  'x': -48.548, 'y': -27.595, 'ax': -48.1,        'ay': -28.3},
    {'name': 'São Paulo',      'x': -46.638, 'y': -23.568, 'ax': -46.3,        'ay': -24.6},
    {'name': 'Aracaju',        'x': -37.073, 'y': -10.947, 'ax': -36.7,        'ay': -11.6},
    {'name': 'Palmas',         'x': -48.333, 'y': -10.184, 'ax': -47.9,        'ay': -10.9},
]

In [ ]:
rows = []
for rgi in results_gdf_low['rgi']:
    try:
        high = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/attributable_mortality_high_negative_raw_{rgi}.npy')
        moderate = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/attributable_mortality_moderate_negative_raw_{rgi}.npy')
        # weights = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/weights_raw_{rgi}.npy')
        ratio = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/moderate_vs_high_ratio_raw_{rgi}.npy')
        factual_moderate = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/factual_mortality_moderate_{rgi}.npy')
        factual_high = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/factual_mortality_high_{rgi}.npy')
        cfactual_moderate = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/counterfactual_mortality_moderate_{rgi}.npy')
        cfactual_high = np.load(f'/bp1/geog-tropical/data/BREATHE/results/raw_arrays_results_100M_mod_high_ratio_p95/counterfactual_mortality_high_{rgi}.npy')
        attributable_moderate = factual_moderate - cfactual_moderate
        attributable_high = factual_high - cfactual_high
        attributable_total = (factual_moderate + factual_high) - (cfactual_moderate + cfactual_high)
    except FileNotFoundError:
        print(f'File not found for RGI: {rgi}')
        continue
    print(f'RGI: {rgi}, high negative: {(high == True).sum()}, moderate negative: {(moderate == True).sum()}, both negative: {((moderate == True) & (high == True)).sum()}')
    

    # but how do we retroactively work out which are negative
    weights_original = generate_weights(models, attributable_total, filter_nans=False)
    p2_5_og, p97_5_og = weighted_percentile(attributable_total, weights_original, [2.5, 97.5])
    mean_og = np.average(attributable_total, weights=weights_original)
    number_of_zeros_weights_original = (weights_original == 0).sum()
    
    weights_nan = generate_weights(models, ratio, filter_nans=True)
    number_of_zeros_weights_nan = (weights_nan == 0).sum()
    
    ratio_filter_double_negative = np.where((((moderate == True) & (high == True))), -ratio, ratio)
    valid = ~np.isfinite(ratio_filter_double_negative)
    values_hr  = np.where(valid, 0, ratio_filter_double_negative)
    
    # this is number of zeros in the ratio array, not the weighted array
    number_of_zeros_ratio = (ratio_filter_double_negative == 0).sum()
    # this is the number of addtional zeros added which is the same as setting the weights to zero
    number_of_zeros_values_hr = (values_hr == 0).sum()
    # this is the actual number of weighted zeros from double negative filtering
    number_of_zeros_added_to_weights_nan = number_of_zeros_values_hr - number_of_zeros_weights_nan
    # total number of zeros in the weighted array is number_of_zeros_weights_nan + number_of_zeros_added_to_weights_nan
    total_number_of_zeros_in_weights_nan = number_of_zeros_weights_nan + number_of_zeros_added_to_weights_nan
    
    # might be crossover in zeros from values_hr and weights_nan
    
    p2_5, p97_5 = weighted_percentile(values_hr, weights_nan, [2.5, 97.5])
    mean = np.average(values_hr, weights=weights_nan)
    median = float(weighted_percentile(values_hr, weights_nan, [50,50])[0])
    print('sum of weights (nan filtered):', weights_nan.sum())
    
    # claculating the mean with the second method
    factual_moderate_mean = np.average(factual_moderate,weights=weights_original)
    factual_moderate_p2_5, factual_moderate_p97_5 = weighted_percentile(factual_moderate, weights_original, [2.5, 97.5])
    factual_high_mean = np.average(factual_high,weights=weights_original)
    factual_high_p2_5, factual_high_p97_5 = weighted_percentile(factual_high, weights_original, [2.5, 97.5])
    attributable_moderate_mean = np.average(attributable_moderate,weights=weights_original)
    attributable_moderate_p2_5, attributable_moderate_p97_5 = weighted_percentile(attributable_moderate, weights_original, [2.5, 97.5])
    attributable_high_mean = np.average(attributable_high,weights=weights_original)
    attributable_high_p2_5, attributable_high_p97_5 = weighted_percentile(attributable_high, weights_original, [2.5, 97.5])
    
    # calculating the mean with nans set to be ignored
    
    
    rows.append({
        'rgi'  : rgi,
        'statistic': 'moderate_high_ratio',
        'mean':      mean,
        'median':    median,
        'p2.5':      p2_5,
        'p97.5':     p97_5,
        # 'bootstrap_mean': bootstrap_mean,
        # 'bootstrap_p2.5': bootstrap_p2_5,
        # 'bootstrap_p97.5': bootstrap_p97_5,
        'factual_high_mean': factual_high_mean,
        'factual_high_p2.5': factual_high_p2_5,
        'factual_high_p97.5': factual_high_p97_5,
        
        'factual_moderate_mean': factual_moderate_mean,
        'factual_moderate_p2.5': factual_moderate_p2_5,
        'factual_moderate_p97.5': factual_moderate_p97_5,
        
        'attributable_high_mean': attributable_high_mean,
        'attributable_high_p2.5': attributable_high_p2_5,
        'attributable_high_p97.5': attributable_high_p97_5,
        
        'attributable_moderate_mean': attributable_moderate_mean,
        'attributable_moderate_p2.5': attributable_moderate_p2_5,
        'attributable_moderate_p97.5': attributable_moderate_p97_5,
        
        'mean_og': mean_og,
        'p2.5_og': p2_5_og,
        'p97.5_og': p97_5_og,
        'number_of_zeros_weights_original': number_of_zeros_weights_original,
        'number_of_zeros_weights_nan': number_of_zeros_weights_nan,
        'number_of_zeros_added_to_weights_nan': number_of_zeros_added_to_weights_nan,
        'total_number_of_zeros_in_weights_nan':total_number_of_zeros_in_weights_nan
        
    })
    
    
    

In [ ]:
# combine with geodata
results = pd.DataFrame(rows)
geo = gpd.read_file(polys)
geo['rgi'] = geo['rgi'].astype(str)

results_gdf = results.merge(geo[['rgi', 'nome_rgi','geometry']], on='rgi', how='left')
results_gdf = gpd.GeoDataFrame(results_gdf, geometry='geometry', crs=geo.crs)
# define gdf_missing as the complement of results_gdf and rgis
gdf_missing = geo[~geo['rgi'].isin(results_gdf['rgi'])]

In [ ]:
# ---- Main figure ----

fig = plt.figure(figsize=(24, 14))
gs  = gridspec.GridSpec(nrows=2, ncols=2, width_ratios=[2.2, 1], wspace=0.05, hspace=0.15)

for col in ['mean', 'attributable_mod_high_ratio', 'factual_mod_high_ratio']:
    if col in results_gdf.columns:
        results_gdf[col] = pd.to_numeric(results_gdf[col], errors='coerce')

# compute the two ratio columns if not already present
results_gdf['attributable_mod_high_ratio'] = (
    results_gdf['attributable_moderate_mean'] / results_gdf['attributable_high_mean']
)
results_gdf['factual_mod_high_ratio'] = (
    results_gdf['factual_moderate_mean'] / results_gdf['factual_high_mean']
)

results_gdf['ratio_second_method'] = (results_gdf['attributable_mod_high_ratio'] / results_gdf['factual_mod_high_ratio'])

# shared colour scale across all three panels
# vmin = np.nanmin(results_gdf[['mean', 'attributable_mod_high_ratio', 'factual_mod_high_ratio']].values)
# vmax = np.nanmax(results_gdf[['mean', 'attributable_mod_high_ratio', 'factual_mod_high_ratio']].values)

panels = [
    (gs[:, 0], 'mean',                        'Ratio of moderate to high heat exposure', 'a.'),
    (gs[0, 1], 'attributable_mod_high_ratio',  'Attributable mortality ratio',            'b.'),
    (gs[1, 1], 'factual_mod_high_ratio',       'Factual mortality ratio',                 'c.'),
]
plt.suptitle('Ratio of moderate to high heat exposure', fontsize=18, fontweight='bold')

# panel a
ax = fig.add_subplot(gs[:, 0])
ax.set_title('Percentage of heat deaths attributable to climate change', fontsize=16, pad=6)
ax.text(-0.02, 1.02, 'a.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig, sm, bounds = plot_brazil_ratio(
    ax, fig, results_gdf,
    statistic='ratio_second_method',
    title=None,
    fontsize=14,
    show_cbar=False,           # suppress per-panel colorbars — added once, shared, below
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')

# panel b
ax = fig.add_subplot(gs[0, 1])
ax.set_title('Heat-related mortality \n attributable to climate change', fontsize=16,  pad=6)
ax.text(-0.02, 1.02, 'b.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig, sm, bounds = plot_brazil_ratio(
    ax, fig, results_gdf,
    statistic='attributable_mod_high_ratio',
    title=None,
    show_cbar=False,           # suppress per-panel colorbars — added once, shared, below
    show_cities=False
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')

# panel c
ax = fig.add_subplot(gs[1, 1])
ax.set_title('Total heat-related mortality', fontsize=16,  pad=6)
ax.text(-0.02, 1.02, 'c.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig, sm, bounds = plot_brazil_ratio(
    ax, fig, results_gdf,
    statistic='factual_mod_high_ratio',
    title=None,
    show_cbar=False,           # suppress per-panel colorbars — added once, shared, below
    show_cities=False
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')





# one shared colorbar for all three panels
 # --- Levels: fine 0–1, coarse 1–10 ---
norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=10)
bounds_low  = np.arange(0, 1.1, 0.1)        # 0.0, 0.1, ..., 1.0
bounds_high = np.arange(2, 11, 1)            # 2, 3, ..., 10
bounds      = np.concatenate([bounds_low, bounds_high])  # skip duplicate 1.0→2
cbar_ax = fig.add_axes([0.25, 0.08, 0.5, 0.015])

cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal',
                    pad=0, extend='both')

# Set ticks at all our bounds
cbar.set_ticks(bounds)

# Label: 1 decimal for 0–1, integer for 1–10
tick_labels = [f'{v:.1f}' for v in bounds_low] + [f'{int(v)}' for v in bounds_high]
cbar.set_ticklabels(tick_labels)

# White line at vcenter=1
# Find where 1.0 sits in normalised (0–1) axes coords
center_pos = norm(1.0)
# cbar.ax.axvline(x=center_pos, color='white', linewidth=2)

# Annotate the two sides
cbar.ax.text(
    0.5, -3.8, 'High > Moderate',
    ha='center', va='top', fontsize=16,
    transform=cbar.ax.transData, color='black'
)
cbar.ax.text(
    5, -3.8, 'Moderate > High',
    ha='center', va='top', fontsize=16,
    transform=cbar.ax.transData, color='black'
)

cbar.set_label("Ratio (moderate / high exposure)", fontsize=20, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=14)

plt.savefig('Figure_3.svg', dpi=300, bbox_inches='tight')

# TODO: why not negative in panels a and b?
# TODO: where are the negatives coming from???

In [ ]:
for col in ['mean', 'attributable_mod_high_ratio', 'factual_mod_high_ratio']:
    if col in results_gdf.columns:
        results_gdf[col] = pd.to_numeric(results_gdf[col], errors='coerce')

results_gdf['attributable_mod_high_ratio'] = (
    results_gdf['attributable_moderate_mean'] / results_gdf['attributable_high_mean']
)
results_gdf['factual_mod_high_ratio'] = (
    results_gdf['factual_moderate_mean'] / results_gdf['factual_high_mean']
)
results_gdf['ratio_second_method'] = (
    results_gdf['attributable_mod_high_ratio'] / results_gdf['factual_mod_high_ratio']
)

norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=10)
bounds_low  = np.arange(0, 1.1, 0.1)
bounds_high = np.arange(2, 11, 1)
bounds      = np.concatenate([bounds_low, bounds_high])
tick_labels = [f'{v:.1f}' for v in bounds_low] + [f'{int(v)}' for v in bounds_high]

In [ ]:
fig_a = plt.figure(figsize=(16.5, 14))
plt.suptitle('Ratio of moderate to high heat exposure', fontsize=18, fontweight='bold')

ax = fig_a.add_subplot(111)
ax.set_title('Percentage of heat deaths attributable to climate change', fontsize=16, pad=6)
ax.text(-0.02, 1.02, 'a.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig_a, sm, bounds = plot_brazil_ratio(
    ax, fig_a, results_gdf,
    statistic='ratio_second_method',
    title=None,
    fontsize=14,
    show_cbar=False,
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')

cbar_ax = fig_a.add_axes([0.25, 0.05, 0.5, 0.02])
cbar = fig_a.colorbar(sm, cax=cbar_ax, orientation='horizontal', pad=0, extend='both')
cbar.set_ticks(bounds)
cbar.set_ticklabels(tick_labels)
cbar.ax.text(0.5, -3.8, 'High > Moderate', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.ax.text(5, -3.8, 'Moderate > High', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.set_label("Ratio (moderate / high exposure)", fontsize=20, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=14)

fig_a.savefig('Figure_3a.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig_b = plt.figure(figsize=(16, 16))
ax = fig_b.add_subplot(111)
ax.set_title('Heat-related mortality \n attributable to climate change', fontsize=16, pad=6)
ax.text(-0.02, 1.02, 'b.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig_b, sm, bounds = plot_brazil_ratio(
    ax, fig_b, results_gdf,
    statistic='attributable_mod_high_ratio',
    title=None,
    show_cbar=False,
    show_cities=False
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')

cbar_ax = fig_b.add_axes([0.25, 0.05, 0.5, 0.02])
cbar = fig_b.colorbar(sm, cax=cbar_ax, orientation='horizontal', pad=0, extend='both')
cbar.set_ticks(bounds)
cbar.set_ticklabels(tick_labels)
cbar.ax.text(0.5, -3.8, 'High > Moderate', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.ax.text(5, -3.8, 'Moderate > High', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.set_label("Ratio (moderate / high exposure)", fontsize=20, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=14)

fig_b.savefig('Figure_3b.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig_c = plt.figure(figsize=(16, 16))
ax = fig_c.add_subplot(111)
ax.set_title('Total heat-related mortality', fontsize=16, pad=6)
ax.text(-0.02, 1.02, 'c.', transform=ax.transAxes,
        fontsize=20, fontweight='bold', va='bottom', ha='right')

ax, fig_c, sm, bounds = plot_brazil_ratio(
    ax, fig_c, results_gdf,
    statistic='factual_mod_high_ratio',
    title=None,
    show_cbar=False,
    show_cities=False
)
gdf_missing.plot(ax=ax, color='none', edgecolor="#bbb6b6", hatch='////', linewidth=0.1)
macro.plot(ax=ax, edgecolor="#FFFFFF", linewidth=1.5, facecolor="none")
ax.set_aspect('auto')
ax.axis('off')

cbar_ax = fig_c.add_axes([0.25, 0.05, 0.5, 0.02])
cbar = fig_c.colorbar(sm, cax=cbar_ax, orientation='horizontal', pad=0, extend='both')
cbar.set_ticks(bounds)
cbar.set_ticklabels(tick_labels)
cbar.ax.text(0.5, -3.8, 'High > Moderate', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.ax.text(5, -3.8, 'Moderate > High', ha='center', va='top', fontsize=16,
             transform=cbar.ax.transData, color='black')
cbar.set_label("Ratio (moderate / high exposure)", fontsize=20, labelpad=10)
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=14)

fig_c.savefig('Figure_3c.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# original weighting should not have any 0
print('number of zeros in original weights:', (results_gdf['number_of_zeros_weights_original']).sum())
print('\n')

# mean number of zeros in weights_nan
print('mean number of zeros in weights_nan:', (results_gdf['number_of_zeros_weights_nan']).mean())
print('number of IGRs with zeros in weights_nan:', (results_gdf['number_of_zeros_weights_nan'] > 0).sum())
print('\n')

# mean number of zeros added to weights_nan due to double negative filtering
print('mean number of zeros added to weights_nan due to double negative filtering:', (results_gdf['number_of_zeros_added_to_weights_nan']).mean())
print('number of IGRs with zeros added to weights_nan due to double negative filtering:', (results_gdf['number_of_zeros_added_to_weights_nan'] > 0).sum())
print('\n')

# mean of total number of zeros in weights_nan
print('mean of total number of zeros in weights_nan:', (results_gdf['total_number_of_zeros_in_weights_nan']).mean())
print('number of IGRs with total number of zeros in weights_nan:', (results_gdf['total_number_of_zeros_in_weights_nan'] > 0).sum())
print('\n')

print('total numbers of IGRs', len(results_gdf))
print('conclusion: weights are different between original and ratio due to nans')


In [ ]:
total_factual = results_gdf['factual_moderate_mean'] + results_gdf['factual_high_mean']
total_attributable = results_gdf['attributable_moderate_mean'] + results_gdf['attributable_high_mean']
original_attributable = results_gdf['mean_og']

# does total_attributable = original_attributable
print('total_attributable:', total_attributable.sum())
print('original_attributable:', original_attributable.sum())
print('difference:', total_attributable.sum() - original_attributable.sum())
print('diff', (total_attributable - original_attributable).sum())